# Cours 10 : attribuer un texte à son auteur avec SuperStyl

Ce notebook reprend la séance en Python : mêmes données, mêmes outils, mais on voit tout au même endroit, avec des graphiques.

Il se lance **depuis le dossier `cours_10`**, avec l'environnement `env` activé (voir [01_installation.md](01_installation.md)) :

```bash
pip install notebook
jupyter notebook cours_10.ipynb
```

Exécutez les cellules dans l'ordre (`Maj + Entrée`).

## 1. Préparer l'environnement

In [ ]:
import sys
sys.path.insert(0, "SuperStyl")   # SuperStyl n'est pas installé : on indique où le trouver

import glob
import pandas as pd
import matplotlib.pyplot as plt
import superstyl as sty

## 2. Le corpus

Trente romans de quatre auteurs. Le nom de chaque fichier commence par l'auteur, suivi d'un `_` : c'est ainsi que SuperStyl sait à qui appartient chaque texte.

In [ ]:
fichiers = sorted(glob.glob("corpus/romans_20s/txt/*.txt"))
auteurs = [f.split("/")[-1].split("_")[0] for f in fichiers]
pd.Series(auteurs).value_counts()

## 3. Extraire les traits

On découpe chaque roman en **échantillons de 2000 mots**, et on compte dans chacun la fréquence des **100 mots les plus fréquents** du corpus.
C'est l'équivalent de la commande :

```bash
python SuperStyl/load_corpus.py -s corpus/romans_20s/txt/*.txt -t words -n 1 -k 100 --sampling --sample_size 2000
```

(environ une minute)

In [ ]:
donnees, traits = sty.load_corpus(data_paths=fichiers, feats="words", n=1, k=100,
                                  sampling=True, units="words", size=2000)
print(donnees.shape)   # (nombre d'échantillons, nombre de colonnes)
donnees.iloc[:5, :8]

Les mots retenus, du plus fréquent au moins fréquent : presque uniquement des **mots grammaticaux**. Les accents ont disparu (conversion en ASCII par défaut).

In [ ]:
print([mot for mot, n in traits[:40]])

## 4. Entraîner et évaluer

`group-k-fold` : on teste chaque roman avec un modèle qui **ne l'a jamais vu** (les échantillons d'un même roman restent ensemble).

In [ ]:
resultats = sty.train_svm(donnees, cross_validate="group-k-fold", get_coefs=True)

La matrice de confusion : en ligne le véritable auteur, en colonne l'auteur prédit.

In [ ]:
resultats["confusion_matrix"]

Les échantillons mal attribués :

In [ ]:
resultats["misattributions"].head(10)

## 5. Pourquoi `group-k-fold` ? Une démonstration

Si l'on mélange les échantillons au hasard avant de les répartir, des passages d'un même roman se retrouvent **à la fois** dans l'entraînement et dans le test.
Le modèle peut alors reconnaître le roman (ses personnages, son vocabulaire) plutôt que l'auteur. On le vérifie avec scikit-learn, la bibliothèque utilisée par SuperStyl.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import KFold, GroupKFold, cross_val_score

X = donnees.drop(columns=["author", "lang"])
y = donnees["author"]
romans = ["_".join(nom.split("_")[:-1]) for nom in donnees.index]   # le roman d'origine de chaque échantillon

modele = make_pipeline(StandardScaler(), Normalizer(), LinearSVC(dual="auto"))
melange = cross_val_score(modele, X, y, cv=KFold(10, shuffle=True, random_state=1)).mean()
groupes = cross_val_score(modele, X, y, cv=GroupKFold(len(set(romans))), groups=romans).mean()
print(f"Échantillons mélangés : {melange:.1%}")
print(f"Romans séparés        : {groupes:.1%}")

Le premier chiffre est **trop beau** : il mesure en partie la reconnaissance du roman, pas de l'auteur. C'est le second qu'il faut publier.

## 6. Quels mots caractérisent chaque auteur ?

Le SVM attribue un poids à chaque mot pour chaque auteur. Les mots au poids le plus élevé sont ceux qui « tirent » vers cet auteur.

In [ ]:
coefs = resultats["coefficients"]
for auteur in coefs.index:
    meilleurs = coefs.loc[auteur].sort_values(ascending=False).index[:10]
    print(f"{auteur:26} {', '.join(meilleurs)}")

Attention aux mots qui ne sont pas du style : un nom de personnage (*frederic* chez Flaubert, s'il apparaît) signale un roman, pas une manière d'écrire.
C'est pourquoi on se limite souvent aux mots les plus fréquents, surtout grammaticaux.

## 7. L'effet de la taille des échantillons

On refait l'expérience avec des échantillons de 500, 1000, 2000 et 5000 mots (environ cinq minutes).

In [ ]:
scores = {}
for taille in [500, 1000, 2000, 5000]:
    d, _ = sty.load_corpus(data_paths=fichiers, feats="words", n=1, k=100,
                           sampling=True, units="words", size=taille)
    X = d.drop(columns=["author", "lang"])
    romans = ["_".join(nom.split("_")[:-1]) for nom in d.index]
    scores[taille] = cross_val_score(modele, X, d["author"], cv=GroupKFold(len(set(romans))), groups=romans).mean()
    print(f"{taille} mots : {len(d)} échantillons, {scores[taille]:.1%}")

In [ ]:
plt.plot(list(scores.keys()), list(scores.values()), marker="o")
plt.xlabel("taille des échantillons (mots)")
plt.ylabel("exactitude (accuracy)")
plt.title("Plus l'échantillon est long, plus l'attribution est sûre")
plt.grid(True)
plt.show()

## 8. Les résultats du benchmark

Si vous avez lancé `bash benchmark.sh` (partie 04), ses résultats se relisent ici.

In [ ]:
import os
if os.path.exists("benchmark/resultats.tsv"):
    bench = pd.read_csv("benchmark/resultats.tsv", sep="\t").fillna("")
    bench["configuration"] = bench["donnees"] + " " + bench["options"]
    bench = bench.sort_values("accuracy")
    plt.barh(bench["configuration"], bench["accuracy"])
    plt.xlabel("exactitude (accuracy)")
    plt.xlim(0.5, 1)
    plt.show()
else:
    print("Lancez d'abord : bash benchmark.sh")

## Exercices

1. Dans la section 3, remplacez `feats="words", n=1, k=100` par `feats="chars", n=3, k=1000` (trigrammes de caractères). Le score change-t-il ? (L'extraction est plus longue : environ trois minutes.)
2. Dans la section 7, refaites la courbe avec `k=300` : plus de mots aide-t-il ?
3. Dans les erreurs de la section 4, quel auteur est le plus souvent confondu avec un autre ? Avez-vous une hypothèse littéraire ?